In [2]:
#INTERACTIVE VERSION!!!!!!!!!!!!!!!
# version 5 integrates new correlation map, also to add help add width/height filtering and cell grid allignments
#this is v5.py with updated volpy fit changes in 5.3 but without multitrial registration components
#version 4 of test_single_trial_RAM_DISK.py with updated MATLAB .mat saving (sped up)
# TO RUN: conda activate caiman
# # python C:\Users\ICNLab\CaImAn_GV\caiman\ICNLAB\test_single_trial_RAM_DISK_5.4_simple.py C:\Users\ICNLab\caiman_data\testdata\testdata\NF107.6B

import argparse
import os
import re
import csv
from datetime import datetime
from pathlib import Path

#froot = "G:/INTRSECT_INVIVO/NPCNF139.5R/20251008/FOV5_T4"
froot = "F:/INTRSECT_INVIVO/NPCNF133.1L/20250911/FOV3_T1"
analysis_mode = "new"  # options: new, old, all


#if re.match(r"^FOV\d+_T\d+$", os.path.basename(froot)):
p = Path(froot)  # normalize to Path
unique_save_string = "-".join(p.parts[-3:])  # last 3 elements
print("Trial folder mode recognized")

# check for prior analysis
#Master CSV path
log_csv_path = Path(froot).parent.parent.parent  / "Analysis" / "MasterAnalysisLOG.csv"

# Ensure the CSV exists with header if needed
if not log_csv_path.exists():
    with open(log_csv_path, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Version", "AnalysisDate", "Trial"])
    print("Created new MasterAnalysisLOG.csv with header.")


existing_trials = set()

if log_csv_path.exists():
    with open(log_csv_path, mode="r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_trials.add(row["Trial"])

trial_exists = unique_save_string in existing_trials

if analysis_mode == "new" and trial_exists:
    print("Mode: new -> skipping", p)
elif analysis_mode == "old" and not trial_exists:
    print("Mode: old -> skipping", p)
else:
    print("Mode: "+str(analysis_mode)+" -> proceeding:", p)
    folder_paths = froot


print("Importing packages and Initializing...")
version="V1.2"
#V1.2: 0.8 corr cutoff, 2 minimum ratio of h over w for spikes, cell_idxs incremented by 1, wheel data appended to mat save
print("version:", version)
import matplotlib
matplotlib.use("qtAgg")   # non-interactive, no windows
print(matplotlib.get_backend())
from base64 import b64encode
import cv2
import glob
import h5py
import imageio
from IPython import get_ipython
from IPython.display import HTML, display, clear_output
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from pathlib import Path
from PIL import Image
import re
import csv
from datetime import datetime

#import to cover extras from single_trial.py
import gc
import scipy.io
from scipy import stats
from scipy.signal import butter, lfilter
from scipy.signal import savgol_filter
import sys
import mat73
import pandas as pd


from pathlib import Path

try:
    cv2.setNumThreads(0)
except:
    pass

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
        get_ipython().run_line_magic('matplotlib', 'qt')
except NameError:
    pass

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.utils.utils import download_demo, download_model
from caiman.source_extraction.volpy import utils
from caiman.source_extraction.volpy.volparams import volparams
from caiman.source_extraction.volpy.volpy import VOLPY
from caiman.source_extraction.volpy.mrcnn import visualize, neurons
import caiman.source_extraction.volpy.mrcnn.model as modellib
from caiman.summary_images import local_correlations_movie_offline
from caiman.summary_images import mean_image
from caiman.paths import caiman_datadir
from caiman.summary_images import local_correlations_movie_in_memory
import gc
from caiman.ICNLAB.single_trial_simple_plotting import plotdata

logging.basicConfig(format=
                    "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s]" \
                    "[%(process)d] %(message)s",
                    level=logging.ERROR)




##BEGIN MAIN ANALYSIS LOOP
#for folder_path in folder_paths:

folder_path = froot

# find the .tsm file in the folder
tsm_files = [f for f in os.listdir(folder_path) if f.endswith(('.tsm', '.dcimg'))]
if not tsm_files:
    print(f"No recording files found in {folder_path}, skipping.")
#continue if more than one .tsm file found
if len(tsm_files) > 1:
    print(f"Multiple recording files found in {folder_path}, skipping.")

fname = os.path.join(folder_path, tsm_files[0])
print('fname is', fname)
print("Processing file:", fname)

fpath = Path(fname)
#Create new unique save name
unique_save_string = "-".join(fpath.parts[-4:-1])
rootpath = str(Path(*fpath.parts[:-4]))+'\\Analysis\\'
print("Unique save string:", unique_save_string)
print("Directory for Analysis Files:", rootpath)
Path(rootpath).mkdir(parents=True, exist_ok=True)
log_csv_path = Path(rootpath) / "MasterAnalysisLOG.csv" #Master CSV path


##
#fname = r'C:\Users\ICNLab\caiman_data\testdata\testdata\FOV1_T2RAM2\FOV1_T2.tsm'
fr = 640  ################################################################REMOVE LATER
print(fname, fr)


##
# Cleanup R:/ drive (temp RAM disk)
print("Cleaning up R:/ drive...")
def safe_close_mmap(arr):
    try:
        if hasattr(arr, "base") and hasattr(arr.base, "close"):
            arr.base.close()
    except Exception as e:
        print("close failed:", e)


# 1. Delete any Python references to memmaps pointing to R:/
try:
    safe_close_mmap(Yr)  # or whatever your memmap object is called
except NameError:
    pass

try:
    safe_close_mmap(mmap_file_rig)  # or whatever your memmap object is called
except NameError:
    pass

gc.collect()  # force Python to release the memory mapping

# 2. Delete all files in R:/
for f in Path(r'R:/').glob('*'):
    if f.is_file():
        f.unlink()
print("Cleared all files from R:/")


##
pw_rigid = False  # flag for pw-rigid motion correction
gsig_filt = (3, 3)  # size of filter, in general gSig (see below),
# change this one if algorithm does not work
max_shifts = (5, 5)  # maximum allowed rigid shift
strides = (48, 48)  # start a new patch for pw-rigid motion correction every x pixels
overlaps = (24, 24)  # overlap between paths (size of patch strides+overlaps)
max_deviation_rigid = 3  # maximum deviation allowed for patch with respect to rigid shifts
border_nan = 'copy'
use_cuda = True

opts_dict = {
    'fnames': fname,
    'fr': fr,
    'pw_rigid': pw_rigid,
    'max_shifts': max_shifts,
    'gSig_filt': gsig_filt,
    'strides': strides,
    'overlaps': overlaps,
    'max_deviation_rigid': max_deviation_rigid,
    'border_nan': border_nan,
    'use_cuda': use_cuda
}

opts = volparams(params_dict=opts_dict)

##
print("Loading data...")
m_orig = cm.load(fname)
ds_ratio = 0.2

##
try:
    c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False)
except:
    print("Cluster running doing restart")
    dview.terminate()
    c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False)
    
##
print("Motion correction...")
mc = MotionCorrect(fname, dview=dview, **opts.get_group('motion'))
mc.motion_correct(save_movie=True, save_dir="R:/")
#about 2.3 minutes for 12800 frames (2m 13-21 s)
print("Done.")

##
print("Loading corrected movie...")
m_rig = cm.load(mc.mmap_file) # 11s
ds_ratio = 0.2
print("Done.")

del m_orig
gc.collect()

Trial folder mode recognized
Mode: new -> skipping F:\INTRSECT_INVIVO\NPCNF133.1L\20250911\FOV3_T1
Importing packages and Initializing...
version: V1.2
qtAgg
fname is F:/INTRSECT_INVIVO/NPCNF133.1L/20250911/FOV3_T1\FOV3_T1.tsm
Processing file: F:/INTRSECT_INVIVO/NPCNF133.1L/20250911/FOV3_T1\FOV3_T1.tsm
Unique save string: NPCNF133.1L-20250911-FOV3_T1
Directory for Analysis Files: F:\INTRSECT_INVIVO\Analysis\
F:/INTRSECT_INVIVO/NPCNF133.1L/20250911/FOV3_T1\FOV3_T1.tsm 640
Cleaning up R:/ drive...
Cleared all files from R:/
Loading data...
Motion correction...
Saving mmap to:  R:/FOV3_T1_rig__d1_512_d2_512_d3_1_order_F_frames_12800.mmap
Done.
Loading corrected movie...


100%|██████████| 1/1 [00:08<00:00,  8.29s/it]


Done.


0

In [3]:
import tifffile

# Example: m_rig is 512x512xN
# Ensure dtype is compatible (uint8, uint16, float32)
print(m_rig.shape, m_rig.dtype)


(12800, 512, 512) float32


In [4]:
import matplotlib.pyplot as plt

plt.imshow(m_rig[0], cmap='gray', aspect='equal')
plt.title("Frame 0")
plt.axis('off')
plt.show()


In [5]:

# Save stacked TIFF
tifffile.imwrite(fname[:-5]+'_stacked_video.tif', m_rig, photometric='minisblack', bigtiff=True)
